In [1]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('Final Dataset.csv')

In [3]:
# Changing columns names for more readability
df.columns = ['village_code', 'visit_month', 'spray_status', 'total_residents',
       'clinic_tests', 'total_rdts', 'avg_distance_to_health_facility_km',
       'incidence_per_1000', 'positivity_rate', 'dist_missing', 'rainfall',
       'rainfall_total', 'humidity', 'temp', 'max_temp', 'min_temp', 'temp_range',
       'prev_rainfall', 'prev2_rainfall', 'prev3_rainfall',
       'prev4_rainfall', 'prev_rainfall_total', 'prev2_rainfall_total',
       'prev3_rainfall_total', 'prev4_rainfall_total', 'prev_humidity',
       'prev2_humidity', 'prev3_humidity', 'prev4_humidity', 'prev_temp', 'prev2_temp',
       'prev3_temp', 'prev4_temp', 'prev_max_temp', 'prev2_max_temp',
       'prev3_max_temp', 'prev4_max_temp', 'prev_min_temp', 'prev2_min_temp',
       'prev3_min_temp', 'prev4_min_temp', 'prev_temp_range', 'prev2_temp_range',
       'prev3_temp_range', 'prev4_temp_range']

ValueError: Length mismatch: Expected axis has 45 elements, new values have 31 elements

In [ ]:
months = df['visit_month'].unique()
months.sort()

split_point = int(len(months) *.8) # This returns the index of the 80th percentile

In [ ]:
# 80% / 20% chronological split
target_columns = ['incidence_per_1000', 'positivity_rate'] # These are the columns our model tries to predict
# And these are poor or useless columns that carry information about which villages have suffered epidemics before
# Causing it to overfit by predicting historical average
useless_columns = ['total_rdts', 'total_residents', 'visit_month', 'avg_distance_to_health_facility_km', 'village_code', 'clinic_tests']
# Train data becomes the majority (80%)
X_train = df[df['visit_month'] < months[split_point]]
y_train = X_train[target_columns]
# Test data becomes the minority (20%)
X_test = df[df['visit_month'] >= months[split_point]]
y_test = X_test[target_columns]
# We finally drop the target columns from the X data
X_train = X_train.drop(columns=target_columns + useless_columns) # We also don't need 'visit_month' anymore it isn't important in training
X_test = X_test.drop(columns=target_columns + useless_columns)

In [ ]:
len(df['avg_distance_to_health_facility_km'].unique()) < len(df['village_code'].unique())
# This surely causes overfitting
# But I think it's supposed to be equal because I imputed some nans with medians and flagged them

In [ ]:
linear = LinearRegression().fit(X_train, y_train)

In [ ]:
mae = mean_absolute_error(y_test, linear.predict(X_test), multioutput='raw_values')
mae
# Linear Regression does have a fair accuracy. I didn't know it can perform this well. I am actually impressed!
# Random Forest should be 10x better!

In [ ]:
model = RandomForestRegressor().fit(X_train, y_train)

In [ ]:
mae = r2_score(y_test, model.predict(X_test), multioutput='raw_values')
mae
# It's accuracy decreased 2x once I dropped the columns that caused it to overfit

In [ ]:
# Let's plot the feature importance and HOPE it's not overfitting
importances = model.feature_importances_
plt.figure(figsize=(20, 8))
plt.bar(X_test.columns, importances)
plt.xlabel('Features'); plt.ylabel('Importances')
plt.title('Feature Importance Bar Chart')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

In [ ]:
y_test['incidence_per_1000'].describe()

In [ ]:
# Let's see if the random forest is shoots at outliers
outlier_sample = np.array(df.drop(columns=useless_columns + target_columns).mean().tolist()) * 100
model.predict(outlier_sample.reshape(1, -1))

In [ ]:
model.predict(outlier_sample.reshape(1, -1))
# This seems to be the maximum it predicts

In [ ]:
df['positivity_rate'].describe()

In [ ]:
# --- Plot 1: Matplotlib Histogram ---
# Create the histogram to visualize the distribution of the 'values' column
plt.figure(figsize=(8, 6))
plt.hist(df['positivity_rate'], bins=30, edgecolor='black', alpha=0.7)
plt.title('Matplotlib Histogram of Values')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df['positivity_rate']

In [ ]:
np.sum((linear.coef_ > 0).astype(int), axis=1)
print((linear.coef_ > 0).astype(int))

In [ ]:
X_train.columns

In [ ]:
target_idx = 0  # choose which target
coefs = pd.DataFrame({
    'feature': X_train.columns,
    'sign': np.sign(linear.coef_[target_idx])
})
coefs